# Code Review Agent — demo end-to-end

Este notebook ejecuta el pipeline completo **sin llave de API** (modo `linter-only`, fallback determinista). Con `ANTHROPIC_API_KEY` en el entorno, el mismo código activa los nodos de Claude (Bug Detector semántico y Comment Synthesizer).

In [2]:
import asyncio, json, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.config import Settings
from src.graph.pipeline import ReviewPipeline, NODE_NAMES
from src.observability import configure_logging
configure_logging('WARNING')
settings = Settings(_env_file=None)
pipeline = ReviewPipeline(settings)
print('mode:', pipeline.mode, '| parser:', pipeline.parser.name, '| scanner:', pipeline.scanner.name)
print('nodes:', ', '.join(NODE_NAMES))

mode: linter-only | parser: tree-sitter | scanner: pattern
nodes: parse_diff, build_context, detect_bugs, check_style, scan_security, review_performance, analyze_test_gaps, synthesize_comments, prioritize


## 1. Un diff con defectos plantados

In [4]:
DIFF = '''diff --git a/app/backup.py b/app/backup.py
--- a/app/backup.py
+++ b/app/backup.py
@@ -1,5 +1,9 @@
-import subprocess
+import os
 
 
-def backup(name):
-    subprocess.run(['tar', 'czf', f'{name}.tgz', name], check=True)
+def backup(name, seen=[]):
+    out = ''
+    for n in seen:
+        out += n
+    try:
+        return os.system('tar czf ' + name + '.tgz ' + name)
+    except:
+        return None
'''
result = asyncio.run(pipeline.run(DIFF, max_comments=10))
print(result.summary)
for c in result.comments:
    print(f'{c.file}:{c.line} [{c.severity}/{c.category}] {c.body}  <{c.rule_id}>')

3 comment(s) across 1 file(s): 1 error, 2 warning, 0 info.
app/backup.py:4 [warning/bug] Mutable default argument is shared across calls.  <static:mutable-default-arg>
app/backup.py:4 [warning/test_gap] `backup` changed and the PR adds no tests.  <test_gap:public-fn-without-tests>
app/backup.py:9 [error/security] `os.system`/`os.popen` run through a shell; user input becomes a command. [CWE-78].  <pattern:shell-injection-system>


## 2. Observabilidad: latencia por nodo, tokens y costo

In [6]:
print(json.dumps(result.metrics.as_dict(), indent=2)[:1200])

{
  "trace_id": "d6c53e07bf3c40feacc3f3d1aae93e22",
  "llm_calls": 0,
  "input_tokens": 0,
  "output_tokens": 0,
  "cost_usd": 0.0,
  "node_timings": [
    {
      "node": "parse_diff",
      "latency_ms": 0,
      "comments_in": 0,
      "comments_out": 0
    },
    {
      "node": "build_context",
      "latency_ms": 0,
      "comments_in": 0,
      "comments_out": 0
    },
    {
      "node": "detect_bugs",
      "latency_ms": 0,
      "comments_in": 0,
      "comments_out": 1
    },
    {
      "node": "review_performance",
      "latency_ms": 0,
      "comments_in": 0,
      "comments_out": 0
    },
    {
      "node": "analyze_test_gaps",
      "latency_ms": 0,
      "comments_in": 0,
      "comments_out": 1
    },
    {
      "node": "scan_security",
      "latency_ms": 10,
      "comments_in": 0,
      "comments_out": 1
    },
    {
      "node": "check_style",
      "latency_ms": 31,
      "comments_in": 0,
      "comments_out": 0
    },
    {
      "node": "synthesize_comment

## 3. Contexto AST (tree-sitter) de los archivos tocados

In [8]:
for ctx in result.contexts:
    print(ctx.path, '| backend:', ctx.parse_backend)
    print('  ', ctx.summary().replace('\n', ' | '))

app/backup.py | backend: tree-sitter
   touched: (module level) | imports: import os


## 4. Evaluación offline reproducible (`python -m eval.run --mode linter-only`)

In [10]:
from src.eval.runner import latest_runs, RUNS_DIR
runs = latest_runs(RUNS_DIR)
for mode, run in runs.items():
    agg = run['aggregate']
    print(f"{mode}: recall={agg['recall']:.1%} precision={agg['precision']:.1%} "
          f"fp_rate={agg['fp_rate']:.1%} cost/PR=${agg['mean_cost_usd']:.4f} "
          f"({run['dataset']['n_cases']} PRs, "
          f"synthetic={run['dataset']['synthetic']})")
print('pending:', [m for m in ('single-pass', 'pipeline') if m not in runs], '-> requieren ANTHROPIC_API_KEY')

linter-only: recall=91.4% precision=97.0% fp_rate=3.0% cost/PR=$0.0000 (38 PRs, synthetic=True)
pending: ['single-pass', 'pipeline'] -> requieren ANTHROPIC_API_KEY
